# Validating ZIP codes against City & State

Goal: for every address row in the R- and C-case address tables, check whether the **ZIP code is actually located in the city & state listed**, so we can separate *genuine city-related ZIPs* from inconsistent ones. This is motivated by a known **city/ZIP time mismatch concentrated in the NxGen period (2011→)**, which is why ZIP was excluded from matching. Here we measure that precisely and produce a per-record validity flag that could rehabilitate ZIP where it *is* consistent.

**Reference crosswalk:** GeoNames US postal dataset (`US.txt`, public domain). For each ZIP it gives the authoritative **state**, a primary **place name**, and a **lat/lon centroid**.

**Why not match city by name alone?** The free GeoNames export lists essentially one place name per ZIP, so name-equality would wrongly reject legitimate alternate / neighbourhood / abbreviated city names. Instead we validate the city **geographically**:

- **State** must match the ZIP's GeoNames state (exact — authoritative and decisive).
- **City** passes if *either* the normalised city equals the ZIP's primary place name, *or* the ZIP's centroid lies within `DIST_KM` of the claimed city's centroid (centroid = mean lat/lon of all ZIPs carrying that city name in that state). When the city is absent from the reference, fall back to a fuzzy name comparison against the ZIP's primary place.

City names are normalised with `norm_city` (expands `st`/`ste`/`mt`/`ft` and directional prefixes, treats hyphens as spaces) applied to **both** sides so legitimate formatting variants don't read as mismatches.

Each row gets a `verdict` and a boolean `zip_valid`:

- `valid` — ZIP consistent with city & state.
- `city_unconfirmed` — state OK and the ZIP is real, but the claimed (small) city isn't in GeoNames, so we can neither confirm nor refute the city. The ZIP is *usually fine*; treated as **unverifiable**, not unreliable.
- `city_mismatch` — state OK but the ZIP is genuinely far (> `DIST_KM`) from the claimed city → **truly unreliable**.
- `state_mismatch` — the ZIP's authoritative state disagrees with the listed state.
- `zip_not_found` — ZIP absent from the reference (retired / territory / malformed).
- `no_zip` — no usable ZIP.

In [ ]:
import zipfile
import numpy as np
import pandas as pd
from pathlib import Path
from rapidfuzz import fuzz

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

# This notebook lives in zip_validation/. Shared merged address tables live in the
# project root (ROOT); the GeoNames input and the zip_validation_*.csv outputs live
# alongside this notebook (HERE).
ROOT = Path("..")
HERE = Path(".")
GEONAMES_ZIP = HERE / "geonames_US.zip"   # downloaded from download.geonames.org/export/zip/US.zip

ADDR_FILES = {
    "R": ROOT / "merged_R_CASES_ADDRESS_with_union_flag.parquet",
    "C": ROOT / "merged_C_CASES_ADDRESS_with_union_flag.parquet",
}

# Tunable thresholds
DIST_KM = 40.0       # ZIP centroid must be within this of the claimed city's centroid
FUZZY_CITY = 85      # fuzzy name fallback when the city is not in the reference

In [2]:
# ---- Helpers ----
def norm_text(s):
    return (s.fillna("").astype(str).str.lower().str.strip()
             .str.replace(r"[.,'\"]", "", regex=True)
             .str.replace(r"\s+", " ", regex=True))

def norm_city(s):
    """City normalisation that matches GeoNames conventions. Applied to BOTH
    the address city and the GeoNames place names so they compare consistently:
      - expand leading 'saint' abbreviations: st->saint, ste->sainte, mt->mount, ft->fort
      - expand leading directionals: n/s/e/w/ne/nw/se/sw -> north/south/... (GeoNames spells them out)
      - treat hyphens as spaces (e.g. 'wilkes-barre' == 'wilkes barre')
    """
    s = norm_text(s)
    # 'saint'/'mount'/'fort' family first (so 'st ' isn't caught by the 's ' directional)
    s = s.str.replace(r"^st\s",  "saint ",  regex=True)
    s = s.str.replace(r"^ste\s", "sainte ", regex=True)
    s = s.str.replace(r"^mt\s",  "mount ",  regex=True)
    s = s.str.replace(r"^ft\s",  "fort ",   regex=True)
    # directionals (two-letter before one-letter)
    s = s.str.replace(r"^ne\s", "northeast ", regex=True)
    s = s.str.replace(r"^nw\s", "northwest ", regex=True)
    s = s.str.replace(r"^se\s", "southeast ", regex=True)
    s = s.str.replace(r"^sw\s", "southwest ", regex=True)
    s = s.str.replace(r"^n\s",  "north ",  regex=True)
    s = s.str.replace(r"^s\s",  "south ",  regex=True)
    s = s.str.replace(r"^e\s",  "east ",   regex=True)
    s = s.str.replace(r"^w\s",  "west ",   regex=True)
    # hyphens -> spaces, collapse
    s = s.str.replace("-", " ", regex=False)
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    return s

def zip5(s):
    """First 5 consecutive digits; '' if none (handles ZIP+4, 'None', malformed)."""
    z = s.astype(str).str.extract(r"(\d{5})", expand=False)
    return z.fillna("")

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0088
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

In [3]:
# ---- Load GeoNames crosswalk and build lookups ----
cols = ["country", "postal_code", "place_name", "admin1", "admin1_code",
        "admin2", "admin2_code", "admin3", "admin3_code", "lat", "lon", "accuracy"]
with zipfile.ZipFile(GEONAMES_ZIP) as z, z.open("US.txt") as f:
    geo = pd.read_csv(f, sep="\t", header=None, names=cols, dtype=str)

geo["lat"] = pd.to_numeric(geo["lat"], errors="coerce")
geo["lon"] = pd.to_numeric(geo["lon"], errors="coerce")
geo["zip5"] = zip5(geo["postal_code"])
geo["state"] = geo["admin1_code"].str.upper().str.strip()
geo["place_norm"] = norm_city(geo["place_name"])   # norm_city on BOTH sides for consistent comparison

# ZIP -> authoritative state, primary place, centroid (one row per ZIP)
zip_ref = (geo.dropna(subset=["lat", "lon"])
              .drop_duplicates("zip5")
              .set_index("zip5")[["state", "place_norm", "lat", "lon"]]
              .rename(columns={"state": "zip_state", "place_norm": "zip_place",
                                "lat": "zip_lat", "lon": "zip_lon"}))

# (state, city) -> centroid, averaged over all ZIPs carrying that city name
city_centroid = (geo.dropna(subset=["lat", "lon"])
                    .groupby(["state", "place_norm"])[["lat", "lon"]].mean()
                    .rename(columns={"lat": "city_lat", "lon": "city_lon"}))

print(f"GeoNames rows: {len(geo):,} | unique ZIPs: {zip_ref.shape[0]:,} | "
      f"(state,city) centroids: {city_centroid.shape[0]:,}")

GeoNames rows: 41,490 | unique ZIPs: 41,488 | (state,city) centroids: 29,542


In [4]:
# ---- Core validation (vectorised) ----
def validate_addresses(addr, case_col):
    d = pd.DataFrame({
        "case_number": addr[case_col].astype(str),
        "data_source": addr["data_source"],
        "raw_zip": addr["zip_code"],
        "zip5": zip5(addr["zip_code"]),
        "state": addr["state"].astype(str).str.upper().str.strip(),
        "city_norm": norm_city(addr["city"]),   # norm_city: matches GeoNames place normalisation
    })

    # attach ZIP reference
    d = d.merge(zip_ref, how="left", left_on="zip5", right_index=True)
    # attach claimed-city centroid
    d = d.merge(city_centroid, how="left",
                left_on=["state", "city_norm"], right_index=True)

    has_zip = d["zip5"] != ""
    zip_found = d["zip_state"].notna()
    state_ok = zip_found & (d["state"] == d["zip_state"])

    # city checks
    name_ok = zip_found & (d["city_norm"] == d["zip_place"]) & (d["city_norm"] != "")

    d["dist_km"] = haversine_km(d["zip_lat"], d["zip_lon"], d["city_lat"], d["city_lon"])
    geo_known = d["city_lat"].notna()   # claimed city present in GeoNames (so distance is meaningful)
    geo_ok = geo_known & (d["dist_km"] <= DIST_KM)

    # fuzzy fallback only where geography can't decide (city not in reference)
    fuzzy_ok = pd.Series(False, index=d.index)
    need_fuzzy = zip_found & ~name_ok & ~geo_ok & ~geo_known & (d["city_norm"] != "")
    if need_fuzzy.any():
        sub = d.loc[need_fuzzy, ["city_norm", "zip_place"]]
        scores = [fuzz.token_sort_ratio(a, b) for a, b in zip(sub["city_norm"], sub["zip_place"])]
        fuzzy_ok.loc[need_fuzzy] = np.array(scores) >= FUZZY_CITY

    city_ok = state_ok & (name_ok | geo_ok | fuzzy_ok)

    # verdict (ordered). The non-city_ok remainder is split:
    #   - geo_known  -> claimed city IS in GeoNames but the ZIP is > DIST_KM away
    #                   => a genuine far mismatch  (city_mismatch, truly unreliable)
    #   - ~geo_known -> claimed city is NOT in GeoNames (small suburb/neighbourhood/base),
    #                   name didn't match and fuzzy failed => we can neither confirm nor
    #                   refute it (city_unconfirmed, unverifiable -- ZIP is usually fine)
    verdict = np.select(
        [~has_zip, ~zip_found, ~state_ok, city_ok, geo_known],
        ["no_zip", "zip_not_found", "state_mismatch", "valid", "city_mismatch"],
        default="city_unconfirmed",
    )
    d["state_ok"] = state_ok
    d["city_ok"] = city_ok
    d["verdict"] = verdict
    d["zip_valid"] = d["verdict"] == "valid"
    return d

results = {}
for side, path in ADDR_FILES.items():
    addr = pd.read_parquet(path)
    case_col = "case_number" if "case_number" in addr.columns else (
        "r_case_number" if side == "R" else "c_case_number")
    results[side] = validate_addresses(addr, case_col)
    print(f"{side}: validated {len(results[side]):,} rows")

R: validated 197,451 rows
C: validated 1,038,762 rows


## 1. Overall verdict distribution

In [5]:
ORDER = ["valid", "city_unconfirmed", "city_mismatch", "state_mismatch", "zip_not_found", "no_zip"]
overall = pd.DataFrame({
    side: r["verdict"].value_counts() for side, r in results.items()
}).reindex(ORDER).fillna(0).astype(int)
overall_pct = (100 * overall / overall.sum()).round(1)
print("Counts:"); display(overall)
print("Percent of rows:"); display(overall_pct)

Counts:


,R,C
verdict,,
valid,162966,869144
city_unconfirmed,9707,41220
city_mismatch,2768,9295
state_mismatch,5737,12466
zip_not_found,5865,28678
no_zip,10408,77959


Percent of rows:


,R,C
verdict,,
valid,82.5,83.7
city_unconfirmed,4.9,4.0
city_mismatch,1.4,0.9
state_mismatch,2.9,1.2
zip_not_found,3.0,2.8
no_zip,5.3,7.5


In [6]:
# Among rows that actually HAVE a usable ZIP, what share is valid?
for side, r in results.items():
    have = r[r["verdict"] != "no_zip"]
    print(f"{side}: of {len(have):,} rows with a ZIP, "
          f"{have['zip_valid'].mean()*100:.1f}% are consistent with city+state "
          f"({(~have['zip_valid']).sum():,} inconsistent)")

R: of 187,043 rows with a ZIP, 87.1% are consistent with city+state (24,077 inconsistent)
C: of 960,803 rows with a ZIP, 90.5% are consistent with city+state (91,659 inconsistent)


## 2. Breakdown by filing system (CHIPS / CATS / NxGen)

This directly tests the observation that the city/ZIP mismatch is concentrated in **NxGen (2011→)**. We report, within rows that have a ZIP, the share that is `valid` per era.

In [7]:
era_order = ["CHIPS", "CATS", "NxGen"]
for side, r in results.items():
    have = r[r["verdict"] != "no_zip"].copy()
    tab = have.groupby("data_source").agg(
        rows_with_zip=("zip_valid", "size"),
        pct_valid=("zip_valid", lambda s: round(100*s.mean(), 1)),
        pct_state_mismatch=("verdict", lambda s: round(100*(s=="state_mismatch").mean(), 1)),
        pct_city_mismatch=("verdict", lambda s: round(100*(s=="city_mismatch").mean(), 1)),
    ).reindex(era_order)
    print(f"=== {side}-side address table: ZIP validity by era ===")
    display(tab)

=== R-side address table: ZIP validity by era ===


,rows_with_zip,pct_valid,pct_state_mismatch,pct_city_mismatch
data_source,,,,
CHIPS,103299,87.3,1.7,1.1
CATS,48957,90.1,1.5,1.0
NxGen,34787,82.4,9.3,3.2


=== C-side address table: ZIP validity by era ===


,rows_with_zip,pct_valid,pct_state_mismatch,pct_city_mismatch
data_source,,,,
CHIPS,460376,87.8,2.3,1.2
CATS,311313,91.4,0.5,1.1
NxGen,189114,95.5,0.1,0.2


In [8]:
# Full verdict x era crosstab (counts) per side
for side, r in results.items():
    print(f"=== {side}: verdict x data_source (counts) ===")
    ct = pd.crosstab(r["verdict"], r["data_source"]).reindex(ORDER).reindex(columns=era_order)
    display(ct.fillna(0).astype(int))

=== R: verdict x data_source (counts) ===


data_source,CHIPS,CATS,NxGen
verdict,,,
valid,90208,44086,28672
city_unconfirmed,5951,2504,1252
city_mismatch,1167,504,1097
state_mismatch,1778,727,3232
zip_not_found,4195,1136,534
no_zip,8990,760,658


=== C: verdict x data_source (counts) ===


data_source,CHIPS,CATS,NxGen
verdict,,,
valid,404112,284467,180565
city_unconfirmed,23032,13776,4412
city_mismatch,5516,3369,410
state_mismatch,10610,1600,256
zip_not_found,17106,8101,3471
no_zip,22724,503,54732


## 3. Inspect mismatches

Eyeball what the inconsistent rows look like — to confirm the checks behave sensibly and to understand the failure modes (wrong state, transposed digits, city far from ZIP, etc.).

In [9]:
SHOW = ["case_number", "data_source", "raw_zip", "zip5", "state", "zip_state",
        "city_norm", "zip_place", "dist_km", "verdict"]
for side, r in results.items():
    print(f"\n##### {side}: sample state_mismatch #####")
    display(r[r["verdict"]=="state_mismatch"][SHOW].head(8))
    print(f"##### {side}: sample city_mismatch (largest distances) #####")
    display(r[r["verdict"]=="city_mismatch"].sort_values("dist_km", ascending=False)[SHOW].head(8))


##### R: sample state_mismatch #####


,case_number,data_source,raw_zip,zip5,state,zip_state,city_norm,zip_place,dist_km,verdict
24,10-RC-340359,NxGen,22101,22101,NC,VA,durham,mc lean,357.611649,state_mismatch
48,05-RC-340119,NxGen,90212,90212,VA,CA,arlington,beverly hills,3700.121881,state_mismatch
60,21-RM-340136,NxGen,97111,97111,CA,OR,claremont,carlton,1326.087662,state_mismatch
66,32-RC-339946,NxGen,60191,60191,CA,IL,antioch,wood dale,2895.956522,state_mismatch
81,19-RC-339912,NxGen,92630,92630,OR,CA,portland,lake forest,1386.971036,state_mismatch
120,01-RC-339606,NxGen,60004,60004,RI,IL,providence,arlington heights,1367.147099,state_mismatch
140,05-RC-339559,NxGen,90212,90212,VA,CA,arlington,beverly hills,3700.121881,state_mismatch
146,12-RM-339466,NxGen,10011,10011,FL,NY,melbourne,new york,1531.564286,state_mismatch


##### R: sample city_mismatch (largest distances) #####


,case_number,data_source,raw_zip,zip5,state,zip_state,city_norm,zip_place,dist_km,verdict
151479,19-RC-013783,CHIPS,99591,99591,AK,AK,anchorage,saint george island,1238.806475,city_mismatch
8820,19-RC-264745,NxGen,99515,99515,AK,AK,ketchikan,anchorage,1238.528418,city_mismatch
30923,16-RC-011012,NxGen,79905,79905,TX,TX,houston,el paso,1073.936574,city_mismatch
13115,16-RC-221123,NxGen,79907,79907,TX,TX,brownsville,el paso,1072.653587,city_mismatch
14795,16-RC-204888,NxGen,79907,79907,TX,TX,brownsville,el paso,1072.653587,city_mismatch
13745,28-RC-215084,NxGen,77002,77002,TX,TX,el paso,houston,1071.655414,city_mismatch
74402,28-RC-006294,CATS,77057,77057,TX,TX,el paso,houston,1059.910925,city_mismatch
74455,28-RC-006347,CATS,75701,75701,TX,TX,el paso,tyler,1043.504890,city_mismatch



##### C: sample state_mismatch #####


,case_number,data_source,raw_zip,zip5,state,zip_state,city_norm,zip_place,dist_km,verdict
654,01-CA-131363,NxGen,02714,02714,RI,MA,east providence,dartmouth,25.821674,state_mismatch
800,04-CA-276054,NxGen,37743,37743,PA,TN,greenville,greeneville,625.003035,state_mismatch
2506,20-CA-183323,NxGen,89521,89521,CA,NV,reno,reno,NaN,state_mismatch
3217,13-CA-110550,NxGen,60426-6021,60426,IN,IL,harvey,harvey,NaN,state_mismatch
4261,22-CA-141955,NxGen,37216,37216,NJ,TN,nashville,nashville,NaN,state_mismatch
4562,22-CA-117137,NxGen,37215,37215,NJ,TN,nashville,nashville,NaN,state_mismatch
5450,12-CA-307462,NxGen,35114,35114,FL,AL,naples,maylene,928.480491,state_mismatch
5749,13-CA-077046,NxGen,60187-0387,60187,LA,IL,wheaton,wheaton,NaN,state_mismatch


##### C: sample city_mismatch (largest distances) #####


,case_number,data_source,raw_zip,zip5,state,zip_state,city_norm,zip_place,dist_km,verdict
691767,19-CA-025510,CHIPS,99546,99546,AK,AK,palmer,adak,1981.071351,city_mismatch
433281,19-CA-030734,CATS,99629,99629,AK,AK,dutch harbor,wasilla,1320.382777,city_mismatch
433457,19-CA-030910,CATS,99692,99692,AK,AK,anchorage,dutch harbor,1277.998956,city_mismatch
430167,19-CA-027591,CATS,99501,99501,AK,AK,ketchikan,anchorage,1241.519503,city_mismatch
694814,19-CB-007055,CHIPS,99591,99591,AK,AK,anchorage,saint george island,1238.806475,city_mismatch
436813,19-CB-009740,CATS,99553,99553,AK,AK,anchorage,akutan,1223.402101,city_mismatch
692767,19-CB-005006,CHIPS,99835,99835,AK,AK,fairbanks,sitka,1093.730335,city_mismatch
687411,19-CA-021062,CHIPS,99707,99707,AK,AK,juneau,fairbanks,1005.684761,city_mismatch


## 4. Unified `zip_reliability` flag across eras

So far the GeoNames check gives a uniform consistency verdict for every era, but the eras differ in what's *available*: NxGen sources carry their own validated, HQ-aware `zip_reliability`; CHIPS/CATS do not. This section produces a single per-case reliability flag that the matcher can consume with one rule, while keeping the **provenance** explicit:

- **NxGen** → use the source flag (`likely_accurate`/`validated_match` → `reliable`; `potentially_hq`/`city_mismatch` → `unreliable`; `missing`). This is the authors' own judgment, which already accounts for HQ ZIPs.
- **CHIPS / CATS** → derive from the GeoNames verdict (`valid` → `reliable`; `state`/`city_mismatch` → `unreliable`; `zip_not_found` → `unverifiable`; `no_zip` → `missing`). This is the *only* signal available for these eras, and it cannot catch a same-metro HQ substitution — so it is a weaker basis.

`reliability_basis` (`source_flag` vs `geonames_consistency`) records which logic was used, and `source_zip_reliability` preserves the raw NxGen flag.

In [ ]:
# ---- Build a unified zip_reliability flag across all eras ----
# NxGen: use the SOURCE-table reliability flag (the authors' own validated, HQ-aware
#        judgment) re-joined from the NxGen source parquets.
# CHIPS/CATS: derive from the GeoNames consistency verdict — the only signal available,
#        since those sources carry no reliability flag and have a single employer
#        address block (no second field to detect HQ substitution).
# `reliability_basis` records which logic produced the value, so the weaker
# CHIPS/CATS basis is never silently equated with NxGen's source judgment.

# nlrb-creating-files is a sibling of the project root, i.e. two levels up from this
# subfolder (zip_validation/ -> project root -> repo parent).
NXGEN_SRC = {
    "R": ROOT / ".." / "nlrb-creating-files" / "NxGen" / "NxGen_DATA" / "R_CASES_ADDRESS_NxGen.parquet",
    "C": ROOT / ".." / "nlrb-creating-files" / "NxGen" / "NxGen_DATA" / "C_CASES_ADDRESS_NxGen.parquet",
}

def standardize_case_number(cn):
    """Match the merge step's key format: 2-digit region, 6-digit docket."""
    parts = str(cn).split("-")
    parts[0] = parts[0].zfill(2)
    parts[-1] = parts[-1].zfill(6)
    return "-".join(parts)

# Source flag value -> unified reliability
SRC_FLAG_MAP = {
    "likely_accurate": "reliable",    # NxGen R: city validation passed
    "potentially_hq":  "unreliable",  # NxGen R: zip may be headquarters
    "validated_match": "reliable",    # NxGen C: extracted city matched
    "city_mismatch":   "unreliable",  # NxGen C: rejected as likely HQ
    "missing":         "missing",
}

# GeoNames verdict -> unified reliability (used for CHIPS/CATS, and any NxGen row
# absent from the source file)
VERDICT_MAP = {
    "valid":            "reliable",
    "city_mismatch":    "unreliable",    # state OK but ZIP genuinely far from claimed city
    "state_mismatch":   "unreliable",
    "city_unconfirmed": "unverifiable",  # state OK, real ZIP, small city absent from GeoNames (ZIP usually fine)
    "zip_not_found":    "unverifiable",  # ZIP absent from the reference (retired/territory/garbage)
    "no_zip":           "missing",
}

def add_unified_reliability(side, r):
    out = r.copy()
    # 1) GeoNames-based default for every row
    out["zip_reliability"] = out["verdict"].map(VERDICT_MAP).astype("object")
    out["reliability_basis"] = "geonames_consistency"
    out["source_zip_reliability"] = pd.NA

    # 2) Overlay the NxGen source flag where available
    src = pd.read_parquet(NXGEN_SRC[side], columns=["case_number", "zip_reliability"]).copy()
    src["case_number"] = src["case_number"].map(standardize_case_number)
    src = src.drop_duplicates("case_number").set_index("case_number")["zip_reliability"]

    is_nx = out["data_source"].eq("NxGen")
    src_for_row = out["case_number"].map(src)         # NaN where no source flag
    use_src = is_nx & src_for_row.notna()

    out.loc[use_src, "source_zip_reliability"] = src_for_row[use_src]
    out.loc[use_src, "zip_reliability"] = src_for_row[use_src].map(SRC_FLAG_MAP)
    out.loc[use_src, "reliability_basis"] = "source_flag"

    out["zip_reliable"] = out["zip_reliability"].eq("reliable")
    return out

results = {side: add_unified_reliability(side, r) for side, r in results.items()}
print("Added unified columns: zip_reliability, reliability_basis, "
      "source_zip_reliability, zip_reliable")

In [11]:
# Unified reliability by era, and the basis used
for side, r in results.items():
    print(f"=== {side}: zip_reliability x era (counts) ===")
    ct = (pd.crosstab(r["zip_reliability"], r["data_source"])
            .reindex(["reliable", "unreliable", "unverifiable", "missing"])
            .reindex(columns=era_order))
    display(ct.fillna(0).astype(int))
    print("reliability_basis:", r["reliability_basis"].value_counts().to_dict())
    print(f"  reliable rows: {r['zip_reliable'].sum():,} / {len(r):,} "
          f"({100*r['zip_reliable'].mean():.1f}%)\n")

# Sanity check: for NxGen, confirm the source flag drove the value (not GeoNames)
for side, r in results.items():
    nx = r[r["data_source"] == "NxGen"]
    print(f"{side} NxGen: {(nx['reliability_basis']=='source_flag').mean()*100:.1f}% "
          f"used the source flag ({(nx['reliability_basis']!='source_flag').sum():,} fell back to GeoNames)")


=== R: zip_reliability x era (counts) ===


data_source,CHIPS,CATS,NxGen
zip_reliability,,,
reliable,90208,44086,27673
unreliable,2945,1231,7114
unverifiable,10146,3640,0
missing,8990,760,658


reliability_basis: {'geonames_consistency': 162006, 'source_flag': 35445}
  reliable rows: 161,967 / 197,451 (82.0%)

=== C: zip_reliability x era (counts) ===


data_source,CHIPS,CATS,NxGen
zip_reliability,,,
reliable,404112,284467,189114
unreliable,16126,4969,45090
unverifiable,40138,21877,0
missing,22724,503,9642


reliability_basis: {'geonames_consistency': 794916, 'source_flag': 243846}
  reliable rows: 877,693 / 1,038,762 (84.5%)

R NxGen: 100.0% used the source flag (0 fell back to GeoNames)
C NxGen: 100.0% used the source flag (0 fell back to GeoNames)


## 5. Export per-case ZIP-validity flags

A slim table keyed by case number, so the matcher (`match_r_to_c_cases.py`) and the multi-R notebook can consume *genuine city-related ZIPs*. Prefer the unified `zip_reliability` / `zip_reliable` (era-aware: NxGen source flag where available, GeoNames consistency otherwise); `verdict` / `zip_valid` remain available as the pure GeoNames result.

In [ ]:
keep = ["case_number", "data_source", "zip5", "state", "zip_state",
        "city_norm", "zip_place", "dist_km", "state_ok", "city_ok",
        "verdict", "zip_valid",
        # unified cross-era reliability
        "zip_reliability", "reliability_basis", "source_zip_reliability", "zip_reliable"]
for side, r in results.items():
    out = HERE / f"zip_validation_{side}_cases.csv"
    r[keep].to_csv(out, index=False)
    print(f"{side}: wrote {len(r):,} rows -> {out.name}  "
          f"({r['zip_valid'].sum():,} GeoNames-valid; "
          f"{r['zip_reliable'].sum():,} unified-reliable)")

## How to read / use this

- **State mismatch** is the cleanest red flag — the ZIP's state is authoritative, so a disagreement means the ZIP (or state) is wrong. Independently cross-validated at ~99.9% against a ZIP-prefix→state map.
- **`city_mismatch` vs `city_unconfirmed`** — both have a correct state, but they mean different things:
  - `city_mismatch` = the claimed city *is* in GeoNames and the ZIP sits > `DIST_KM` away → a genuine mismatch (truly unreliable).
  - `city_unconfirmed` = the claimed (small) city isn't in GeoNames at all, so distance can't be computed; the ZIP is real and in the right state and is *usually fine* — we just can't confirm the specific city name (a limit of GeoNames' one-place-per-ZIP export, not a bad ZIP). Mapped to **unverifiable**.
- The **era breakdown** (Section 2) quantifies the NxGen city/ZIP problem directly.
- **Unified flag (Section 4)** is the recommended thing to consume downstream:
  - `reliable` (high precision — validated) → safe to use the ZIP.
  - `unreliable` (`state_mismatch` + true `city_mismatch`) → don't use the ZIP.
  - `unverifiable` (`city_unconfirmed` + `zip_not_found`) → ZIP can't be verified; lean usable for `city_unconfirmed` (state is right), more cautious for `zip_not_found`. Use `verdict` to tell them apart.
  - `missing` → no ZIP.
  - `reliability_basis` marks NxGen (`source_flag`, HQ-aware) vs CHIPS/CATS (`geonames_consistency`, weaker — can't catch a same-metro HQ ZIP).
- **Reuse:** join `zip_validation_{R,C}_cases.csv` on case number. In the multi-R analysis, gate the *same city, diff zip* branch test on `zip_reliable` so it only fires where the ZIP is genuine.